In [38]:
import pandas as pd
import re

records = []

with open("/home/nanopore/projects/amplicon_workflow/resources/custom_dbs/silva_v138.2/species_taxid.fasta", "r") as f:
    for line in f:
        if line.startswith(">"):
            taxid = re.search(r'^>(\d+):', line)
            accession = re.search(r"\['([A-Z]+\d+)\.", line)
            
        if taxid and accession:
            records.append({
                "tax_id": taxid.group(1),
                "accession": accession.group(1)
            })

fasta_df = pd.DataFrame(records).drop_duplicates()
fasta_df.head()


,tax_id,accession
0,1,AY846372
2,2,CS394307
4,3,BD359735
6,4,HL196896
8,5,AB000390


In [35]:
bsl_df = pd.read_csv("/home/nanopore/projects/amplicon_workflow/resources/custom_dbs/bsl_table.csv")
bsl_df['accession'] = bsl_df['name'].str.extract(r'ACC\s+(\w+)')
bsl_df = bsl_df[["accession", "bsl", "name"]]

bsl_df.head()


,accession,bsl,name
0,JX133166,bsl_1,Cronobacter sakazakii(ACC JX133166)
1,JX912524,bsl_1,Campylobacter rectus(ACC JX912524)
2,JX912526,bsl_1,Campylobacter sputorum biovar sputorum(ACC JX9...
3,JX912523,bsl_1,Campylobacter mucosalis(ACC JX912523)
4,AY188350,bsl_1,Streptococcus downei(ACC AY188350)


In [42]:
comb_df = bsl_df.set_index("accession").join(fasta_df.set_index("accession")).reset_index().dropna()
comb_df = comb_df[["tax_id","bsl"]]
comb_df["tax_id"] = comb_df["tax_id"].astype(int)
comb_df.head()

,tax_id,bsl
0,342983,bsl_1
1,87336,bsl_1
2,87137,bsl_1
3,85809,bsl_1
4,61981,bsl_1


In [44]:
tax_df = pd.read_csv("/home/nanopore/projects/amplicon_workflow/resources/custom_dbs/silva_v138.2/taxonomy.tsv", sep="\t")
tax_df = tax_df[["tax_id", "species"]]


df = comb_df.set_index("tax_id").join(tax_df.set_index("tax_id")).reset_index()


df.head()



,tax_id,bsl,species
0,342983,bsl_1,Cronobacter_sakazakii
1,87336,bsl_1,Campylobacter_rectus
2,87137,bsl_1,Campylobacter_sputorum
3,85809,bsl_1,Campylobacter_mucosalis
4,61981,bsl_1,Streptococcus_downei


In [ ]:
emu_res = pd.read_csv("/home/nanopore/projects/amplicon_workflow/results/16S_compost_SUP_SILVA_097/emu/emu-combined-species.tsv", sep="\t")

df = df[["species", "bsl"]]
df["species"] = df["species"].str.replace("_", " ")


final_df = df.set_index("species").join(emu_res.set_index("species")).reset_index()

final_df.tail()


,species,bsl,genus,family,order,class,phylum,superkingdom,barcode71,barcode17,...,barcode24,barcode05,barcode34,barcode66,barcode25,barcode18,barcode79,barcode91,barcode84,barcode87
1562,Nocardia farcinica,bsl_2,Nocardia,Nocardiaceae,Corynebacteriales,Actinobacteria,Actinobacteriota,Bacteria,NaN,0.001228,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000541,NaN
1663,Klebsiella pneumoniae,bsl_2,Klebsiella,Enterobacteriaceae,Enterobacterales,Gammaproteobacteria,Proteobacteria,Bacteria,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.00136,NaN,NaN,NaN,NaN
1980,Anaplasma phagocytophilum,bsl_2,Anaplasma,Anaplasmataceae,Rickettsiales,Alphaproteobacteria,Proteobacteria,Bacteria,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023,Legionella londiniensis,bsl_2,Legionella,Legionellaceae,Legionellales,Gammaproteobacteria,Proteobacteria,Bacteria,NaN,0.001310,...,NaN,0.005149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2506,Nocardia paucivorans,bsl_2,Nocardia,Nocardiaceae,Corynebacteriales,Actinobacteria,Actinobacteriota,Bacteria,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000455,NaN


In [58]:
final_df.to_csv("/home/nanopore/projects/amplicon_workflow/results/16S_compost_SUP_SILVA_097/emu/emu_combined_species_bsl.csv", index=False)